# Speaker augmentation on TESS (cross-speaker splits)

Same setup as `TESS_emotion.ipynb`: add the TESS dataset (`ejlok1/toronto-emotional-speech-set-tess`), GPU T4 x1, Internet on, Save & Run All. About 1 hour. Download `TESS_aug_results.zip` from the Output tab.

In [ ]:
CODE_B64 = '...'
CODE_BUILD = '2026-11-17 18:53 d3fb5e5a'

In [ ]:
import base64, io, os, sys, tarfile, subprocess, time, torch
PROJ = '/kaggle/working/project'
os.makedirs(PROJ, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode='r:gz').extractall(PROJ)
print('code build:', CODE_BUILD)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (enable a GPU!)')
import transformers, librosa, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| librosa',
      librosa.__version__, '| sklearn', sklearn.__version__)
wavs = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f.lower().endswith('.wav')]
print(len(wavs), 'wav files under /kaggle/input, e.g.', wavs[:2])
assert wavs, 'TESS not found: add the dataset (Add data -> ejlok1/toronto-emotional-speech-set-tess)'

In [ ]:
def run(cmd):
    print('+', cmd, flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, cwd=PROJ)
    print(f'({time.time() - t0:.0f} s)')
    assert p.returncode == 0, f'failed: {cmd}'

FEAT = '/tmp/tess_features_aug'
run(f'python scripts/extract_augmented.py --tess /kaggle/input --out {FEAT}')

In [ ]:
run(f'python scripts/run_all.py --features {FEAT} --results Results_aug --seeds 0 1 2 --epochs 40 --splits oaf2yaf yaf2oaf --variants speech_mean speech_mean_aug speech_logmel speech_logmel_aug')

In [ ]:
run('python scripts/make_results.py --results Results_aug')
print(open(f'{PROJ}/Results_aug/accuracy_table.md').read())

In [ ]:
import glob, zipfile
keep = {f'Results_aug/runs/{v}/{s}/seed0/model.pt' for v in ('speech_mean', 'speech_mean_aug')
        for s in ('oaf2yaf', 'yaf2oaf')}
out = '/kaggle/working/TESS_aug_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{PROJ}/Results_aug/**/*', recursive=True):
        rel = os.path.relpath(p, PROJ).replace(os.sep, '/')
        if os.path.isfile(p) and (not rel.endswith('model.pt') or rel in keep):
            z.write(p, rel)
    for f in ('metadata.json', 'splits.json'):
        z.write(f'{FEAT}/{f}', f'Results_aug/data/{f}')
print(out, round(os.path.getsize(out) / 1e6, 1), 'MB | code build', CODE_BUILD)